# 🎙️ Iraqi Voice Studio — RVC v2 Full Interactive Studio on Google Colab
تم تطوير هذا الدفتر لتدريب واستنساخ الصوت العراقي بأعلى جودة ودقة متناهية عبر بنية RVC v2 مع لوحة تحكم تفاعلية ومراقبة متدفقة واستديو تحويل مباشر.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/drive/MyDrive/RVC_Iraqi
!mkdir -p /content/drive/MyDrive/RVC_Iraqi/datasets

In [ ]:
# @title 📥 جلب وتجهيز حزمة الصوت والبيانات (Dataset Downloader & Uploader)
import os, sys, glob, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

out_ds = widgets.Output()

ds_radio = widgets.RadioButtons(
    options=[
        ("⚡ مجموعة الصوت العراقي الافتراضية الموصى بها (Voice.zip)", "default"),
        ("📤 رفع ملف مضغوط (ZIP) من جهازك الحاسوب", "upload"),
        ("📁 استخدام مسار مخصص في Google Drive", "drive"),
    ],
    value="default",
    description="مصدر البيانات:",
    layout=widgets.Layout(width="95%")
)

drive_path_text = widgets.Text(
    value="/content/drive/MyDrive/RVC_Iraqi/Voice.zip",
    description="مسار الملف:",
    layout=widgets.Layout(width="95%")
)

zip_uploader = widgets.FileUpload(
    accept=".zip",
    multiple=False,
    description="📂 اختر ملف ZIP",
    layout=widgets.Layout(width="280px")
)

btn_unpack = widgets.Button(
    description="📥 جلب وفك ضغط البيانات الصوتية (Download & Unpack)",
    button_style="success",
    icon="download",
    layout=widgets.Layout(width="380px")
)

def on_ds_radio_change(change):
    if ds_radio.value == "drive":
        drive_path_text.layout.display = "block"
        zip_uploader.layout.display = "none"
    elif ds_radio.value == "upload":
        drive_path_text.layout.display = "none"
        zip_uploader.layout.display = "block"
    else:
        drive_path_text.layout.display = "none"
        zip_uploader.layout.display = "none"

drive_path_text.layout.display = "none"
zip_uploader.layout.display = "none"
ds_radio.observe(on_ds_radio_change, names="value")

def on_unpack_clicked(b):
    with out_ds:
        clear_output(wait=True)
        choice = ds_radio.value
        if choice == "default":
            print("⚡ جاري تنزيل مجموعة الصوت العراقي الافتراضية (Voice.zip)...")
            url = "https://drive.usercontent.google.com/download?id=1dXino3_rN6uV6RUGJpyfo3U4_bxxBXs8&export=download&authuser=0&confirm=t"
            cmd = f'wget -q --show-progress -O /content/Voice.zip "{url}"'
            os.system(cmd)
        elif choice == "upload":
            if not zip_uploader.value:
                print("❌ يرجى اختيار ملف ZIP من جهازك أولاً!")
                return
            for fname, item in zip_uploader.value.items():
                print(f"جاري حفظ {fname}...")
                Path("/content/Voice.zip").write_bytes(item["content"])
        else:
            p = Path(drive_path_text.value.strip())
            if not p.exists():
                print(f"❌ لم يتم العثور على الملف في المسار: {p}")
                return
            shutil.copy(p, "/content/Voice.zip")

        print("📦 جاري فك الضغط وتنظيم المقاطع الصوتية في /content/Voice...")
        os.system("unzip -q -o /content/Voice.zip -d /content/")
        os.system("rm -f /content/Voice/merged_audio.mp3")

        # حصر الملفات
        v_dir = Path("/content/Voice")
        if v_dir.exists():
            files = list(v_dir.glob("*.ogg")) + list(v_dir.glob("*.wav")) + list(v_dir.glob("*.mp3"))
            print(f"🎉 تم استخراج {len(files):,} مقطع صوتي نقي بنجاح في /content/Voice!")
        else:
            print("⚠️ تعذر التحقق من مجلد /content/Voice.")

btn_unpack.on_click(on_unpack_clicked)

display(widgets.VBox([
    widgets.HTML("<h4>📥 تحديد واستيراد بيانات الصوت (Dataset Import):</h4>"),
    ds_radio,
    drive_path_text,
    zip_uploader,
    widgets.HBox([btn_unpack]),
    out_ds
]))

with out_ds:
    v_dir = Path("/content/Voice")
    files = list(v_dir.glob("*.ogg")) + list(v_dir.glob("*.wav")) if v_dir.exists() else []
    print(f"📁 الحالة الحالية: يتوفر {len(files)} مقطع صوتي جاهز في /content/Voice.")


In [ ]:
# @title 🎧 استعراض مقاطع مجموعة البيانات (Dataset Audio Player & Preview)
import os, glob
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio

out_preview = widgets.Output()

def get_files():
    v = Path("/content/Voice")
    if not v.exists(): return [("لا توجد ملفات", "")]
    files = list(v.glob("*.ogg")) + list(v.glob("*.wav")) + list(v.glob("*.mp3"))
    return [(f.name, str(f)) for f in sorted(files)] or [("لا توجد ملفات", "")]

dropdown_files = widgets.Dropdown(
    options=get_files(),
    description="اختر مقطعًا:",
    layout=widgets.Layout(width="80%")
)

btn_refresh = widgets.Button(description="🔄 تحديث القائمة", button_style="info", layout=widgets.Layout(width="180px"))

def on_select(change):
    with out_preview:
        clear_output(wait=True)
        path = dropdown_files.value
        if path and os.path.exists(path):
            print(f"🔊 تشغيل المقطع الصوتي: {Path(path).name}")
            display(Audio(path, rate=48000))

dropdown_files.observe(on_select, names="value")

def on_ref(b):
    dropdown_files.options = get_files()

btn_refresh.on_click(on_ref)

display(widgets.VBox([
    widgets.HTML("<h4>🎧 استمع إلى أي مقطع من مجموعة البيانات:</h4>"),
    widgets.HBox([dropdown_files, btn_refresh]),
    out_preview
]))

# تشغيل أول مقطع تلقائياً
with out_preview:
    if dropdown_files.options and dropdown_files.options[0][1]:
        p = dropdown_files.options[0][1]
        print(f"🔊 تشغيل المقطع: {Path(p).name}")
        display(Audio(p, rate=48000))


In [ ]:
!curl -fsSL https://raw.githubusercontent.com/Alqudimi/VoiceCall/main/scripts/colab/install_colab.sh | bash

In [ ]:
# @title 0) 🔑 تنزيل القواعد المسبقة وتسجيل الدخول إلى Hugging Face
import os, sys, subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

out_auth = widgets.Output()

base_dropdown = widgets.Dropdown(
    options=[
        ("⭐ titan (48k v2 - الأقوى والأعلى جودة للصوت العراقي)", "titan"),
        ("⚡ ov2 (40k v2 - قاعدة سريعة وخفيفة)", "ov2"),
        ("📦 stock (40k v1 - قاعدة RVC الأصلية)", "stock"),
    ],
    value="titan",
    description="القاعدة المسبقة:",
    layout=widgets.Layout(width="95%")
)

hf_token_text = widgets.Password(
    value="",
    description="توكن HF:",
    placeholder="hf_...",
    layout=widgets.Layout(width="95%")
)

btn_login_download = widgets.Button(
    description="🔑 تسجيل الدخول وتنزيل القواعد (Download Bases & Login)",
    button_style="primary",
    icon="download",
    layout=widgets.Layout(width="420px")
)

def on_login_download_clicked(b):
    with out_auth:
        clear_output(wait=True)
        token = hf_token_text.value.strip()
        base = base_dropdown.value
        os.environ["HF_TOKEN"] = token

        # تسجيل الدخول إلى HuggingFace
        if token:
            try:
                from huggingface_hub import login, whoami
                login(token=token, add_to_git_credential=False)
                user = whoami()["name"]
                print(f"✅ تم تسجيل الدخول إلى Hugging Face بنجاح باسم: {user}")
            except Exception as e:
                print(f"⚠️ خطأ في تسجيل الدخول إلى Hugging Face: {e}")

        # تنزيل القواعد المسبقة
        print(f"📥 جاري التحقق وتنزيل القاعدة المسبقة ({base})...")
        cmd = f"cd /content/iraqi-voice-platform && bash scripts/colab/get_pretrained.sh {base} 48k"
        ret = os.system(cmd)
        if ret == 0:
            print("✅ تم تنزيل وتثبيت القواعد المسبقة ونماذج HuBERT / RMVPE بنجاح!")
        else:
            print(f"⚠️ حدث خطأ أثناء التنزيل (رمز الخروج: {ret}).")

btn_login_download.on_click(on_login_download_clicked)

display(widgets.VBox([
    widgets.HTML("<h4>🔑 إعداد الحساب والقواعد المسبقة للتدريب:</h4>"),
    base_dropdown,
    hf_token_text,
    widgets.HBox([btn_login_download]),
    out_auth
]))

with out_auth:
    print("✅ الإعدادات جاهزة. اضغط على الزر أعلاه لتسجيل الدخول وتنزيل القواعد.")


In [ ]:
# @title 1) 📁 فحص وتجهيز مجموعة البيانات (Data Check, Preprocessing & FAISS Indexing)
os.environ["UV_NO_SYNC"] = "1"
import os, sys, glob, subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# ═══════════════ إعدادات البيانات ═══════════════
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
DATASET_PATH = "/content/Voice"  # @param {type:"string"}
SR = "48k"  # @param ["32k", "40k", "48k"]
F0_METHOD = "rmvpe"  # @param ["rmvpe", "fcpe", "pm", "crepe"]
DENOISE = False  # @param {type:"boolean"} (تنظيف طيفي اختياري للبيانات الصاخبة)
LOUDNESS_NORM = True  # @param {type:"boolean"} (توحيد جهارة المقاطع - موصى به)

out = widgets.Output()

def get_audio_files(folder):
    p = Path(folder)
    if not p.exists(): return []
    return list(p.glob("*.wav")) + list(p.glob("*.ogg")) + list(p.glob("*.mp3")) + list(p.glob("*.flac"))

def on_check_clicked(b):
    with out:
        clear_output(wait=True)
        print("🔍 جاري فحص بيانات الصوت في:", DATASET_PATH)
        files = get_audio_files(DATASET_PATH)
        print(f"📊 عدد الملفات الصوتية المكتشفة: {len(files):,}")
        if not files:
            print(f"❌ لم يتم العثور على ملفات صوتية في {DATASET_PATH}!")
            return
        cmd = f"cd /content/iraqi-voice-platform && uv run rvc-platform data-check '{DATASET_PATH}'"
        os.system(cmd)

def on_prep_clicked(b):
    with out:
        clear_output(wait=True)
        print(f"⚙️ جاري بدء التجهيز الكامل لمجموعة البيانات (SR={SR}, F0={F0_METHOD})...")
        denoise_flag = "--denoise" if DENOISE else "--no-denoise"
        loudness_flag = "--loudness" if LOUDNESS_NORM else "--no-loudness"
        cmd = (f"cd /content/iraqi-voice-platform && uv run rvc-platform prepare-dataset "
               f"'{DATASET_PATH}' '{EXPERIMENT}' --sr '{SR}' --f0-method '{F0_METHOD}' "
               f"--speech-filter 0.25 {denoise_flag} {loudness_flag}")
        os.system(cmd)

btn_check = widgets.Button(description="🔍 فحص جودة البيانات (Data Check)", button_style="info", icon="search", layout=widgets.Layout(width="280px"))
btn_prep = widgets.Button(description="⚙️ تجهيز البيانات والفهرس (Prepare & Index)", button_style="primary", icon="cogs", layout=widgets.Layout(width="320px"))

btn_check.on_click(on_check_clicked)
btn_prep.on_click(on_prep_clicked)

display(widgets.HBox([btn_check, btn_prep]), out)

# فحص مبدئي سريع عند التشغيل
with out:
    files = get_audio_files(DATASET_PATH)
    run_dir = Path(f"/content/iraqi-voice-platform/runs/{EXPERIMENT}")
    print("=" * 65)
    print(f"🎙️ حالة مجلد البيانات: {DATASET_PATH}")
    print(f"   • الملفات المتاحة: {len(files):,} مقطع صوتي")
    if run_dir.exists():
        index_files = list(run_dir.glob("*.index"))
        features = list(run_dir.glob("3_feature*"))
        print(f"   • حالة التجهيز السابقة: جاهزة ومستخرجة في {run_dir.name}")
        if index_files:
            print(f"   • الفهرس الصوتي: {index_files[-1].name}")
    print("=" * 65)
    print("💡 يمكنك الضغط على أحد الأزرار أعلاه للفحص المفصل أو إعادة التجهيز في أي وقت.")


In [ ]:
# @title 2) 🚀 لوحة التحكم في التدريب (Training Control Panel: Start, Stop, Resume)
# env gpu: preflight check to ensure CUDA torch is preserved
os.environ["UV_NO_SYNC"] = "1"
import os, sys, time, json, signal, subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

# ═══════════════ إعدادات التدريب الموصى بها (RVC v2 48k) ═══════════════
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
DATASET_PATH = "/content/Voice"  # @param {type:"string"}
EPOCHS = 300  # @param {type:"integer"}
BATCH = 12  # @param {type:"integer"}
LR = 0.0001  # @param {type:"number"}
SAVE_EVERY = 10  # @param {type:"integer"}
F0_METHOD = "rmvpe"  # @param ["rmvpe", "fcpe", "pm", "crepe"]
SR = "48k"  # @param ["32k", "40k", "48k"]

# رافعات الجودة والبحث (State-of-the-art)
EMA_DECAY = 0.999  # @param {type:"number"}
STFT_WEIGHT = 2.0  # @param {type:"number"}
SPK_WEIGHT = 1.0  # @param {type:"number"}
VAL_SPLIT = 0.10  # @param {type:"number"}
SAVE_BEST = True  # @param {type:"boolean"}
AUGMENT = True  # @param {type:"boolean"}
WARMUP = 1  # @param {type:"integer"}
COSINE = 0.1  # @param {type:"number"}
CLIP_GRAD_NORM = 500  # @param {type:"number"}
HF_REPO = "eng7mi/Iraqi-v2"  # @param {type:"string"}

out_train = widgets.Output()

def get_trainer_pids():
    try:
        res = subprocess.check_output(["pgrep", "-f", "rvc-platform train start"]).decode().strip()
        pids = [int(p) for p in res.split() if p.isdigit()]
        return pids
    except Exception:
        return []

def build_train_command(resume=True):
    flags = [
        f"--experiment '{EXPERIMENT}'",
        f"--dataset '{DATASET_PATH}'",
        f"--sr {SR}",
        f"--version v2",
        f"--epochs {EPOCHS}",
        f"--batch {BATCH}",
        f"--lr {LR}",
        f"--save-every {SAVE_EVERY}",
        f"--f0-method '{F0_METHOD}'",
        f"--ema-decay {EMA_DECAY}",
        f"--stft-weight {STFT_WEIGHT}",
        f"--spk-weight {SPK_WEIGHT}",
        f"--val-split {VAL_SPLIT}",
        "--save-best" if SAVE_BEST else "--no-save-best",
        "--augment" if AUGMENT else "--no-augment",
        f"--warmup {WARMUP}",
        f"--cosine {COSINE}",
        f"--clip-grad-norm {CLIP_GRAD_NORM}",
    ]
    if HF_REPO.strip():
        flags.append(f"--hf-repo '{HF_REPO.strip()}'")
    if not resume:
        flags.append("--clean-start")
    return "cd /content/iraqi-voice-platform && RVC_TORCH_COMPILE=0 uv run rvc-platform train start " + " ".join(flags)

def on_start_clicked(b):
    with out_train:
        clear_output(wait=True)
        pids = get_trainer_pids()
        if pids:
            print(f"⚠️ التدريب يعمل بالفعل حاليًا (PID: {pids})! لا تقم بتشغيله مجددًا لتفادي تعارض الذاكرة.")
            on_status_clicked(None)
            return
        print("🚀 جاري بدء التدريب من البداية...")
        cmd = build_train_command(resume=False)
        subprocess.Popen(cmd, shell=True)
        time.sleep(2)
        on_status_clicked(None)

def on_stop_clicked(b):
    with out_train:
        clear_output(wait=True)
        pids = get_trainer_pids()
        if not pids:
            print("ℹ️ لا توجد جلسة تدريب قيد التشغيل حاليًا لإيقافها.")
            return
        print(f"⏹️ جاري إرسال إشارة إيقاف تدريجي وآمن للتدريب (PID: {pids})...")
        os.system("cd /content/iraqi-voice-platform && uv run rvc-platform train stop || true")
        for pid in pids:
            try:
                os.kill(pid, signal.SIGTERM)
            except Exception:
                pass
        time.sleep(2)
        print("✅ تم إيقاف عملية التدريب بأمان مع حفظ آخر نقطة تحقق.")

def on_resume_clicked(b):
    with out_train:
        clear_output(wait=True)
        pids = get_trainer_pids()
        if pids:
            print(f"⚠️ التدريب يعمل بالفعل حاليًا (PID: {pids})! لا حاجة لاستئنافه.")
            on_status_clicked(None)
            return
        print("🔄 جاري استئناف التدريب تلقائيًا من آخر نقطة تحقق...")
        cmd = build_train_command(resume=True)
        subprocess.Popen(cmd, shell=True)
        time.sleep(2)
        on_status_clicked(None)

def on_status_clicked(b):
    with out_train:
        clear_output(wait=True)
        pids = get_trainer_pids()
        prog_file = Path(f"/content/iraqi-voice-platform/runs/{EXPERIMENT}/progress.json")
        print("=" * 65)
        print(f"🎙️ حالة جلسة التدريب: {EXPERIMENT}")
        print("=" * 65)
        if pids:
            print(f"الحالة:        🟢 قيد التشغيل الفعلي في الخلفية (PIDs: {pids})")
        else:
            print("الحالة:        ⚪ متوقف حاليًا")
        if prog_file.exists():
            try:
                prog = json.loads(prog_file.read_text())
                cur_ep = prog.get("epoch", 0)
                tot_ep = prog.get("total_epochs", 300)
                best = prog.get("best", {})
                print(f"التقدم الحالي: {cur_ep}/{tot_ep} حقبة (الخطوة: {prog.get('global_step', 0):,})")
                if best:
                    print(f"أفضل نموذج:    الحقبة {best.get('epoch')} (الدرجة: {best.get('score', 0):.4f})")
            except Exception:
                pass
        print("-" * 65)
        # GPU check
        try:
            gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu", "--format=csv,noheader"]).decode().strip()
            print(f"بطاقة GPU:     {gpu_info}")
        except Exception:
            pass
        print("=" * 65)

btn_start = widgets.Button(description="🚀 بدء التدريب (Start)", button_style="success", icon="play", layout=widgets.Layout(width="200px"))
btn_stop = widgets.Button(description="⏹️ إيقاف تدريجي (Stop)", button_style="danger", icon="stop", layout=widgets.Layout(width="200px"))
btn_resume = widgets.Button(description="🔄 استئناف (Resume)", button_style="warning", icon="history", layout=widgets.Layout(width="200px"))
btn_status = widgets.Button(description="ℹ️ فحص الحالة (Status)", button_style="info", icon="info", layout=widgets.Layout(width="200px"))

btn_start.on_click(on_start_clicked)
btn_stop.on_click(on_stop_clicked)
btn_resume.on_click(on_resume_clicked)
btn_status.on_click(on_status_clicked)

display(widgets.HBox([btn_start, btn_stop, btn_resume, btn_status]), out_train)

# عرض الحالة المبدئية عند فتح الخلية
on_status_clicked(None)


In [ ]:
# @title 📝 مراقبة سجلات التدريب الفورية (Live Training Console Logs)
# تشغيل هذه الخلية يعرض أحدث السجلات الناتجة عن عملية التدريب مباشرة
!tail -n 15 /content/iraqi-voice-platform/runs/iraqi_voice/events.jsonl


In [ ]:
# @title 3) 📊 لوحة المراقبة الحية المتدفقة ورسم المنحنيات (RVC Live Streaming Dashboard)
import os, sys, json, time, glob, threading
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# ═══════════════ إعدادات المراقبة ═══════════════
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}
STREAM_INTERVAL = 4  # @param {type:"integer"} (فترة التحديث بالثواني أثناء البث)
PLOT_GRAPHS = True  # @param {type:"boolean"} (عرض المخططات البيانية)

RUN_DIR = Path(f"/content/iraqi-voice-platform/runs/{EXPERIMENT}")
progress_file = RUN_DIR / "progress.json"
events_file = RUN_DIR / "events.jsonl"

out_dash = widgets.Output()
status_label = widgets.Label(value="الحالة: جاهز (اضغط 'بدء البث المباشر' للتحديث التلقائي المستمر)")

# قفل وأعلام التحكم بالخيط
_stream_thread = None
_stop_event = threading.Event()

def render_snapshot():
    if not progress_file.exists():
        print(f"⏳ بانتظار بدء الجلسة في {RUN_DIR}... لا توجد بيانات مسجلة بعد.")
        return

    try:
        prog = json.loads(progress_file.read_text())
    except Exception as e:
        print(f"جاري قراءة البيانات ({e})...")
        return

    cur_epoch = prog.get("epoch", 0)
    total_epochs = prog.get("total_epochs", 300)
    global_step = prog.get("global_step", 0)
    state = prog.get("state", "running")
    losses = prog.get("latest_losses", {})
    best = prog.get("best", {})

    last_epoch_sec = 68.0
    val_events = []
    prog_steps = []
    prog_mel = []
    prog_stft = []

    if events_file.exists():
        with open(events_file) as f:
            for line in f:
                if not line.strip(): continue
                try:
                    ev = json.loads(line)
                    top = ev.get("topic")
                    if top == "training.epoch_done":
                        last_epoch_sec = float(ev.get("epoch_seconds", last_epoch_sec))
                    elif top == "training.validation":
                        val_events.append(ev)
                    elif top == "training.progress":
                        prog_steps.append(ev.get("global_step", 0))
                        ls = ev.get("latest_losses", {})
                        prog_mel.append(min(float(ls.get("mel", 0)), 100.0))
                        prog_stft.append(min(float(ls.get("stft", 0)), 50.0))
                except Exception:
                    pass

    pct = (cur_epoch / total_epochs) * 100 if total_epochs else 0
    bar_len = 25
    filled = int(bar_len * cur_epoch / total_epochs) if total_epochs else 0
    bar = "█" * filled + "░" * (bar_len - filled)

    rem_epochs = max(0, total_epochs - cur_epoch)
    eta_min = (rem_epochs * last_epoch_sec) / 60

    print("=" * 68)
    print(f"🎙️  لوحة متابعة تدريب الصوت العراقي (RVC v2 48k) — {EXPERIMENT}")
    print("=" * 68)
    status_icon = "🟢 قيد التدريب الفعلي الآن" if state == "running" else f"⚪ {state}"
    print(f"الحالة:             {status_icon}")
    print(f"التقدم الإجمالي:    [{bar}] {cur_epoch}/{total_epochs} حقبة ({pct:.1f}%)")
    print(f"الخطوة التراكمية:   {global_step:,} (خطوات الحقبة: {prog.get('steps_per_epoch', 63)})")
    print(f"زمن الحقبة الواحدة: {last_epoch_sec:.1f} ثانية | الوقت المتبقي المقدر: ~{eta_min:.0f} دقيقة ({eta_min/60:.1f} ساعة)")
    print("-" * 68)
    print("📊 الخسائر اللحظية (Losses):")
    print(f"   • Mel Loss:       {losses.get('mel', 0):.4f}")
    print(f"   • STFT Loss:      {losses.get('stft', 0):.4f}")
    print(f"   • Generator Loss: {losses.get('gen', 0):.4f}")
    print(f"   • Discrim Loss:   {losses.get('disc', 0):.4f}")
    print(f"   • Learning Rate:  {losses.get('lr', 0):.2e}")
    print("-" * 68)
    if best:
        print(f"🏆 أفضل نموذج محفوظ حتى الآن (Best Checkpoint):")
        print(f"   • تم تسجيله في الحقبة: Epoch {best.get('epoch')} (الخطوة {best.get('global_step')})")
        print(f"   • درجة الجودة المركبة: {best.get('score', 0):.4f} (كلما قل الرقم زادت الجودة والشبه)")
    print("=" * 68)

    if val_events:
        print("\n📋 سجل التقييم الموضوعي لأحدث الحقب:")
        print(f"{'الحقبة':<6} | {'خسارة Mel':<12} | {'شبه المتحدث ECAPA':<18} | {'F0 RMSE':<10} | {'الدرجة المركبة':<12}")
        print("-" * 68)
        for v in val_events[-5:]:
            ep = v.get("epoch", 0)
            sc = v.get("scores", {})
            spk_pct = sc.get("val_spk", 0) * 100
            print(f"{ep:<6} | {sc.get('val_mel', 0):<12.4f} | {sc.get('val_spk', 0):<6.4f} ({spk_pct:.1f}%)    | {sc.get('val_f0_rmse', 0):<10.2f} | {sc.get('composite', 0):<12.4f}")

    if PLOT_GRAPHS and len(prog_steps) > 5 and len(val_events) > 1:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.2))
        ax1.plot(prog_steps, prog_mel, label="Mel Loss", color="#1f77b4", linewidth=1.5)
        ax1.plot(prog_steps, prog_stft, label="Multi-res STFT", color="#ff7f0e", linewidth=1.5)
        ax1.set_title("Training Loss Convergence", fontsize=11, fontweight="bold")
        ax1.set_xlabel("Global Step")
        ax1.set_ylabel("Loss")
        ax1.grid(True, linestyle="--", alpha=0.5)
        ax1.legend(loc="upper right")

        val_eps = [v.get("epoch") for v in val_events]
        val_comps = [v.get("scores", {}).get("composite", 0) for v in val_events]
        val_spks = [v.get("scores", {}).get("val_spk", 0) * 100 for v in val_events]
        ax2.plot(val_eps, val_comps, label="Composite (أقل أفضل)", color="#2ca02c", marker="o", markersize=3)
        ax2_spk = ax2.twinx()
        ax2_spk.plot(val_eps, val_spks, label="Speaker Sim % (أعلى أفضل)", color="#9467bd", marker="s", markersize=3, linestyle="--")
        ax2.set_title("Validation Quality & Speaker Similarity", fontsize=11, fontweight="bold")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Composite Score", color="#2ca02c")
        ax2_spk.set_ylabel("ECAPA Similarity (%)", color="#9467bd")
        ax2.grid(True, linestyle="--", alpha=0.5)
        lines1, labels1 = ax2.get_legend_handles_labels()
        lines2, labels2 = ax2_spk.get_legend_handles_labels()
        ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

        plt.tight_layout()
        plt.show()

def _stream_worker():
    while not _stop_event.is_set():
        with out_dash:
            clear_output(wait=True)
            render_snapshot()
        time.sleep(STREAM_INTERVAL)

def on_start_stream(b):
    global _stream_thread
    if _stream_thread is not None and _stream_thread.is_alive():
        status_label.value = "الحالة: 🔴 البث المباشر يعمل بالفعل!"
        return
    _stop_event.clear()
    status_label.value = f"الحالة: 🔴 البث المباشر يعمل الآن (تحديث كل {STREAM_INTERVAL} ثوانٍ)..."
    _stream_thread = threading.Thread(target=_stream_worker, daemon=True)
    _stream_thread.start()

def on_stop_stream(b):
    _stop_event.set()
    status_label.value = "الحالة: ⏹️ تم إيقاف البث المباشر بنجاح."

def on_refresh_now(b):
    with out_dash:
        clear_output(wait=True)
        render_snapshot()
    status_label.value = "الحالة: 🔄 تم التحديث الفوري."

btn_start_stream = widgets.Button(description="🔴 تشغيل البث الحي (Live Stream)", button_style="danger", icon="play", layout=widgets.Layout(width="240px"))
btn_stop_stream = widgets.Button(description="⏹️ إيقاف البث (Stop Stream)", button_style="warning", icon="stop", layout=widgets.Layout(width="220px"))
btn_refresh_now = widgets.Button(description="🔄 تحديث يدوي فوري (Refresh)", button_style="info", icon="refresh", layout=widgets.Layout(width="220px"))

btn_start_stream.on_click(on_start_stream)
btn_stop_stream.on_click(on_stop_stream)
btn_refresh_now.on_click(on_refresh_now)

display(widgets.VBox([
    widgets.HBox([btn_start_stream, btn_stop_stream, btn_refresh_now]),
    status_label,
    out_dash
]))

# عرض المشهد الأولي فور فتح الخلية
with out_dash:
    render_snapshot()


In [ ]:
# @title 4) ☁️ المزامنة التلقائية والرفع إلى Hugging Face في خيط منفصل (Background HF Sync)
import os, sys, time, json, subprocess, threading
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

REPO_ID = "eng7mi/Iraqi-v2"  # @param {type:"string"}
EXPERIMENT = "iraqi_voice"  # @param {type:"string"}

out_hf = widgets.Output()
status_hf_label = widgets.Label(value="الحالة: اضغط على أحد الأزرار للمزامنة أو فحص المستودع")

def is_daemon_running():
    try:
        res = subprocess.check_output(["pgrep", "-f", "hf_auto_sync.py"]).decode().strip()
        return bool(res)
    except Exception:
        return False

def on_instant_sync_clicked(b):
    with out_hf:
        clear_output(wait=True)
        print("🚀 جاري بدء المزامنة الفورية في خيط منفصل في الخلفية...")
        status_hf_label.value = "الحالة: ⏳ جاري رفع الملفات إلى Hugging Face..."

        def _upload_task():
            cmd = f"python3 /content/iraqi-voice-platform/scripts/hf_auto_sync.py --once"
            ret = os.system(cmd)
            with out_hf:
                if ret == 0:
                    print("✅ تمت المزامنة بنجاح! تم رفع النموذج والفهرس v2 والإعدادات.")
                    status_hf_label.value = "الحالة: ✅ تم الرفع بنجاح."
                else:
                    print("⚠️ حدث خطأ أثناء المزامنة. راجع السجل أدناه.")
                    status_hf_label.value = "الحالة: ⚠️ حدث خطأ أثناء الرفع."
                on_check_repo_clicked(None)

        t = threading.Thread(target=_upload_task, daemon=True)
        t.start()

def on_check_repo_clicked(b):
    with out_hf:
        clear_output(wait=True)
        daemon_status = "🟢 نشط ويعمل في الخلفية (يراقب كل 60 ثانية)" if is_daemon_running() else "⚪ غير مشغل"
        print("=" * 65)
        print(f"☁️ مستودع النماذج: https://huggingface.co/{REPO_ID}")
        print(f"خيط الرفع التلقائي: {daemon_status}")
        print("=" * 65)

        # قراءة حالة المزامنة المحلية
        state_file = Path(f"/content/iraqi-voice-platform/runs/{EXPERIMENT}/hf_synced_state.json")
        if state_file.exists():
            try:
                st = json.loads(state_file.read_text())
                print("آخر ملف تم رفعه: ", st.get("last_synced_file"))
                print("أفضل نموذج في best/:", st.get("best_file"))
                synced_time = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(st.get("synced_at", 0)))
                print("وقت آخر مزامنة:  ", synced_time)
            except Exception:
                pass
        print("-" * 65)

        # استعلام الملفات الفعلية من Hugging Face Hub
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            files = api.list_repo_files(repo_id=REPO_ID, repo_type="model")
            print(f"📦 الملفات المحفوظة فعليًا داخل المستودع ({len(files)} ملف):")
            for f in sorted(files):
                if f.startswith("best/") or f.endswith(".pth") or f.endswith(".index") or f.endswith(".wav"):
                    print(f"   • {f}")
        except Exception as e:
            print(f"تعذر الاستعلام من HuggingFace API ({e})")
        print("=" * 65)

btn_sync_now = widgets.Button(description="🚀 مزامنة فورية الآن (Sync to HF)", button_style="success", icon="cloud-upload", layout=widgets.Layout(width="280px"))
btn_check_repo = widgets.Button(description="🔍 فحص ملفات المستودع (Check Repo)", button_style="info", icon="list", layout=widgets.Layout(width="280px"))

btn_sync_now.on_click(on_instant_sync_clicked)
btn_check_repo.on_click(on_check_repo_clicked)

display(widgets.VBox([
    widgets.HBox([btn_sync_now, btn_check_repo]),
    status_hf_label,
    out_hf
]))

# عرض مبدئي للحالة
on_check_repo_clicked(None)


In [ ]:
# @title 5) 🎙️ استديو تحويل واستنساخ الصوت المتقدم (Audio-to-Audio Voice Studio)
os.environ["UV_NO_SYNC"] = "1"
import os, sys, glob, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Audio, Javascript

EXPERIMENT = "iraqi_voice"

# ═══════════════ دوال فحص النماذج والفهارس والعينات ═══════════════
def scan_models():
    exp_dir = Path(f"/content/iraqi-voice-platform/runs/{EXPERIMENT}")
    exports = exp_dir / "exports"
    models = []
    if exports.exists():
        best = sorted(exports.glob("*best*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
        regular = sorted(exports.glob("*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
        for b in best:
            models.append((f"⭐ [أفضل نموذج]: {b.name}", str(b)))
        for r in regular:
            if "best" not in r.name:
                models.append((f"📦 [حقبة تدريب]: {r.name}", str(r)))
    models_dir = Path("/content/iraqi-voice-platform/models")
    if models_dir.exists():
        for m in models_dir.glob("*.pth"):
            if not any(str(m) == val for _, val in models):
                models.append((f"🔗 [الرابط العام]: {m.name}", str(m)))
    return models or [("⚠️ لا توجد نماذج بعد", "")]

def scan_indices():
    exp_dir = Path(f"/content/iraqi-voice-platform/runs/{EXPERIMENT}")
    indices = []
    if exp_dir.exists():
        for idx in sorted(exp_dir.glob("*.index"), key=lambda p: p.stat().st_mtime, reverse=True):
            tag = "⭐ [v2 768-dim - موصى به]" if "v2" in idx.name else "📦 [v1 256-dim]"
            indices.append((f"{tag} {idx.name}", str(idx)))
    return indices or [("⚠️ لا يوجد فهرس", "")]

def scan_audio_files():
    files = []
    # 1. الملفات المرفوعة مؤخرًا
    for up in sorted(Path("/content").glob("upload_*.*")):
        files.append((f"📤 [ملف مرفوع]: {up.name}", str(up)))
    # 2. ملفات مجلد العينات
    v = Path("/content/Voice")
    if v.exists():
        for ext in ["*.ogg", "*.wav", "*.mp3", "*.m4a", "*.flac"]:
            for f in sorted(v.glob(ext)):
                files.append((f"📁 [عينة]: {f.name}", str(f)))
    return files or [("لا توجد ملفات صوتية", "")]

# ═══════════════ عناصر واجهة المستخدم (Widgets) ═══════════════
source_tabs = widgets.Tab()

# التبويب 1: رفع ملف من الحاسوب
file_uploader = widgets.FileUpload(
    accept='audio/*',
    multiple=False,
    description='📂 اختر ملف صوتي',
    layout=widgets.Layout(width='280px')
)
upload_via_colab_btn = widgets.Button(
    description='☁️ رفع عبر نافذة المتصفح (Colab)',
    button_style='info',
    icon='upload',
    layout=widgets.Layout(width='280px')
)
upload_status = widgets.Label(value="اختر ملفًا صوتيًا (WAV, MP3, OGG, M4A) لرفعه واستنساخه مباشرة.")
tab_upload = widgets.VBox([widgets.HBox([file_uploader, upload_via_colab_btn]), upload_status])

# التبويب 2: اختيار من العينات
sample_dropdown = widgets.Dropdown(
    options=scan_audio_files(),
    description='اختر عينة:',
    layout=widgets.Layout(width='95%')
)
tab_sample = widgets.VBox([sample_dropdown])

# التبويب 3: مسار يدوي
custom_path_text = widgets.Text(
    value='/content/Voice/07c7df904ddd452484b0d20c71a0a753.ogg',
    placeholder='/path/to/your/audio.mp3',
    description='المسار:',
    layout=widgets.Layout(width='95%')
)
tab_custom = widgets.VBox([custom_path_text])

source_tabs.children = [tab_upload, tab_sample, tab_custom]
source_tabs.set_title(0, '📤 رفع صوت من جهازك')
source_tabs.set_title(1, '📁 اختيار من العينات')
source_tabs.set_title(2, '✏️ كتابة مسار يدوي')

# قوائم النموذج والفهرس
model_dropdown = widgets.Dropdown(options=scan_models(), description='النموذج:', layout=widgets.Layout(width='95%'))
index_dropdown = widgets.Dropdown(options=scan_indices(), description='الفهرس:', layout=widgets.Layout(width='95%'))

# أشرطة التمرير والخيارات
pitch_slider = widgets.IntSlider(value=0, min=-12, max=12, step=1, description='النغمة (نصف درجة):', layout=widgets.Layout(width='70%'))
btn_preset_male = widgets.Button(description='👨 رجالي (0)', layout=widgets.Layout(width='120px'))
btn_preset_female = widgets.Button(description='👩 أنثوي (+12)', layout=widgets.Layout(width='120px'))
btn_preset_deep = widgets.Button(description='🧔 تخشين (-12)', layout=widgets.Layout(width='120px'))

def set_pitch(val): pitch_slider.value = val
btn_preset_male.on_click(lambda b: set_pitch(0))
btn_preset_female.on_click(lambda b: set_pitch(12))
btn_preset_deep.on_click(lambda b: set_pitch(-12))

pitch_box = widgets.VBox([
    pitch_slider,
    widgets.HBox([widgets.Label(value="اختصارات سريعة:"), btn_preset_male, btn_preset_female, btn_preset_deep])
])

index_rate_slider = widgets.FloatSlider(value=0.75, min=0.0, max=1.0, step=0.05, description='قوة الفهرس (0.75 الأفضل):', layout=widgets.Layout(width='70%'))
protect_slider = widgets.FloatSlider(value=0.33, min=0.0, max=0.5, step=0.01, description='حماية الحروف (0.33):', layout=widgets.Layout(width='70%'))

btn_refresh_lists = widgets.Button(description='🔄 تحديث قوائم النماذج والملفات', button_style='info', icon='refresh', layout=widgets.Layout(width='280px'))
btn_convert = widgets.Button(description='🎙️ تحويل واستنساخ الصوت الآن', button_style='success', icon='play-circle', layout=widgets.Layout(width='320px', height='45px'))

out_studio = widgets.Output()

# ═══════════════ معالجة الأحداث (Event Handlers) ═══════════════
uploaded_filepath = None

def on_file_uploaded(change):
    global uploaded_filepath
    if not file_uploader.value: return
    for fname, item in file_uploader.value.items():
        content = item['content']
        ext = Path(fname).suffix or ".wav"
        save_path = Path(f"/content/upload_{int(time.time())}{ext}")
        save_path.write_bytes(content)
        uploaded_filepath = str(save_path)
        upload_status.value = f"✅ تم استلام وحفظ: {fname} ({len(content)/(1024*1024):.2f} MB)"
        # تحديث قائمة العينات
        sample_dropdown.options = scan_audio_files()
        with out_studio:
            clear_output(wait=True)
            print(f"🎵 تم رفع الملف الصوتي بنجاح: {fname}")
            display(Audio(uploaded_filepath, rate=48000))

file_uploader.observe(on_file_uploaded, names='value')

def on_upload_colab_clicked(b):
    global uploaded_filepath
    try:
        from google.colab import files
        with out_studio:
            clear_output(wait=True)
            print("⏳ الرجاء اختيار ملف صوتي من نافذة الرفع المنبثقة...")
            up = files.upload()
            for fn, data in up.items():
                ext = Path(fn).suffix or ".wav"
                p = Path(f"/content/upload_{int(time.time())}{ext}")
                p.write_bytes(data)
                uploaded_filepath = str(p)
                upload_status.value = f"✅ تم رفع: {fn}"
                sample_dropdown.options = scan_audio_files()
                print(f"✅ تم رفع {fn} بنجاح!")
                display(Audio(uploaded_filepath, rate=48000))
    except Exception as e:
        upload_status.value = f"تعذر استخدام نافذة كولاب ({e})"

upload_via_colab_btn.on_click(on_upload_colab_clicked)

def on_refresh_clicked(b):
    model_dropdown.options = scan_models()
    index_dropdown.options = scan_indices()
    sample_dropdown.options = scan_audio_files()
    with out_studio:
        clear_output(wait=True)
        print("🔄 تم تحديث قوائم النماذج والفهارس والملفات الصوتية بنجاح.")

btn_refresh_lists.on_click(on_refresh_clicked)

def on_convert_clicked(b):
    with out_studio:
        clear_output(wait=True)
        # تحديد الملف المدخل بناءً على التبويب المختار
        selected_tab = source_tabs.selected_index
        input_audio = None
        if selected_tab == 0:
            input_audio = uploaded_filepath
            if not input_audio or not os.path.exists(input_audio):
                print("❌ لم تقم برفع أي ملف صوتي بعد! استخدم زر الرفع أولاً أو اختر من التبويب الثاني.")
                return
        elif selected_tab == 1:
            input_audio = sample_dropdown.value
        else:
            input_audio = custom_path_text.value.strip()

        if not input_audio or not os.path.exists(input_audio):
            print(f"❌ الملف الصوتي غير موجود: {input_audio}")
            return

        model_path = model_dropdown.value
        index_path = index_dropdown.value
        if not model_path or not os.path.exists(model_path):
            print("❌ يرجى اختيار نموذج صالح من القائمة المنسدلة.")
            return

        output_wav = "/content/converted_voice.wav"
        pitch = pitch_slider.value
        index_rate = index_rate_slider.value
        protect = protect_slider.value

        print("=" * 68)
        print("🎙️ جاري تحويل واستنساخ الصوت عبر RVC v2...")
        print(f"   • الملف المدخل:  {Path(input_audio).name}")
        print(f"   • النموذج:       {Path(model_path).name}")
        if index_path and os.path.exists(index_path):
            print(f"   • الفهرس الصوتي: {Path(index_path).name} (قوة الفهرس: {index_rate})")
        print(f"   • درجة النغمة:   {pitch} نصف درجة")
        print("=" * 68)

        index_arg = f"--index '{index_path}' --index-rate {index_rate}" if (index_path and os.path.exists(index_path)) else ""
        cmd = (f"cd /content/iraqi-voice-platform && uv run rvc-platform inference "
               f"'{input_audio}' {EXPERIMENT} -o '{output_wav}' "
               f"--pitch {pitch} {index_arg} --filter-radius 3 --protect {protect}")

        ret = os.system(cmd)
        if ret == 0 and os.path.exists(output_wav):
            size_mb = os.path.getsize(output_wav) / (1024 * 1024)
            print(f"\n🎉 تم التحويل بنجاح فائق! (الحجم: {size_mb:.2f} MB)")
            print("\n🎧 1) استمع للصوت الأصلي المدخل:")
            display(Audio(str(input_audio), rate=48000))
            print("\n🎙️ 2) استمع للصوت المستنسخ بالعراقية:")
            display(Audio(output_wav, rate=48000))
            print("\n📥 رابط تنزيل الملف المحوّل:")
            print(f"   مسار الملف على السيرفر: {output_wav}")
        else:
            print(f"\n⚠️ تعذر التحويل (رمز الخطأ: {ret}). يرجى التأكد من مسار الملف.")

btn_convert.on_click(on_convert_clicked)

# تجميع الواجهة الكاملة
main_ui = widgets.VBox([
    widgets.HTML("<h3>🎙️ استديو تحويل الصوت المباشر — RVC Voice Cloning Studio</h3>"),
    widgets.HTML("<b>1) مصدر الصوت المراد استنساخه:</b>"),
    source_tabs,
    widgets.HTML("<hr><b>2) إعدادات النموذج والفهرس:</b>"),
    widgets.HBox([btn_refresh_lists]),
    model_dropdown,
    index_dropdown,
    widgets.HTML("<hr><b>3) ضبط النغمة ونقل النبرة:</b>"),
    pitch_box,
    index_rate_slider,
    protect_slider,
    widgets.HTML("<hr>"),
    widgets.HBox([btn_convert]),
    out_studio
])

display(main_ui)

with out_studio:
    print("✅ الاستديو جاهز تمامًا! يمكنك رفع صوتك أو اختيار عينة ثم الضغط على 'تحويل واستنساخ الصوت الآن'.")


In [ ]:
# @title 6) 🌐 تشغيل واجهة المستخدم الرسومية مع رابط خارجي (Web UI & Public Tunnel)
os.environ["UV_NO_SYNC"] = "1"
# يقوم هذا الأمر بتشغيل خادم الاستوديو التفاعلي وتوليد رابط خارجي (Cloudflare / Localtunnel)
# للوصول إلى لوحة التحكم من أي متصفح أو هاتف محمول.
import os

print("=" * 65)
print("🌐 جاري تشغيل خادم الاستوديو التفاعلي RVC Web Studio...")
print("⚠️ سيعمل هذا الخادم في هذه الخلية حتى تضغط زر الإيقاف.")
print("=" * 65)

!cd /content/iraqi-voice-platform && uv run rvc-platform web --host 0.0.0.0 --tunnel
